# Lab Experiment 5 : Linear Regression through Gradient Descent
#### MCA-A
#### 2547112

## Aim
To implement Linear Regression using the Gradient Descent optimization algorithm and evaluate its performance on the **Student Performance** dataset (UCI Machine Learning Repository).

## Dataset
- **Name:** Student Performance Dataset
- **Source:** https://archive.ics.uci.edu/dataset/320/student+performance
- **File used:** `student-mat.csv` (Math course), 395 students, 33 attributes
- **Target:** `G3` - final grade (0-20)


## 1. Import Libraries

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

## 2. Load the Dataset

In [6]:
DATA_PATH = "student-mat.csv"

df = pd.read_csv(DATA_PATH, sep=";")

print(f"Shape: {df.shape}")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'student-mat.csv'

## 3. Data Preprocessing

### 3.1 Check for Missing Values

In [ ]:
print(df.isnull().sum().sum(), "missing values in the dataset")
df.info()

### 3.2 Encode Categorical Variables

The dataset has three kinds of categorical columns:
- **Binary yes/no** columns (`schoolsup`, `famsup`, `paid`, `activities`, `nursery`, `higher`, `internet`, `romantic`) -> mapped to 0/1
- **Binary nominal** columns (`school`, `sex`, `address`, `famsize`, `Pstatus`) -> mapped to 0/1
- **Multi-class nominal** columns (`Mjob`, `Fjob`, `reason`, `guardian`) -> one-hot encoded

In [ ]:
df_enc = df.copy()

binary_yes_no = ["schoolsup", "famsup", "paid", "activities",
                  "nursery", "higher", "internet", "romantic"]
for col in binary_yes_no:
    df_enc[col] = df_enc[col].map({"yes": 1, "no": 0})

binary_map = {
    "school": {"GP": 0, "MS": 1},
    "sex": {"F": 0, "M": 1},
    "address": {"U": 0, "R": 1},
    "famsize": {"LE3": 0, "GT3": 1},
    "Pstatus": {"T": 0, "A": 1},
}
for col, mapping in binary_map.items():
    df_enc[col] = df_enc[col].map(mapping)

nominal_cols = ["Mjob", "Fjob", "reason", "guardian"]
df_enc = pd.get_dummies(df_enc, columns=nominal_cols, drop_first=True)

# Convert any remaining booleans (from get_dummies) to int
bool_cols = df_enc.select_dtypes(include="bool").columns
df_enc[bool_cols] = df_enc[bool_cols].astype(int)

print(f"Shape after encoding: {df_enc.shape}")
df_enc.head()

## 4. Feature Selection & Target Variable

`G3` (final grade) is the target. All remaining columns are used as input features,
including the period grades `G1` and `G2` (as provided by the dataset).

In [ ]:
X = df_enc.drop(columns=["G3"]).values.astype(float)
y = df_enc["G3"].values.astype(float)

feature_names = df_enc.drop(columns=["G3"]).columns.tolist()

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

## 5. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train set: {X_train.shape}, Test set: {X_test.shape}")

## 6. Feature Scaling

The scaler is fit only on the training data and then used to transform both
the training and test sets, to avoid data leakage.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Add bias (intercept) column
X_train_bias = np.c_[np.ones((X_train_scaled.shape[0], 1)), X_train_scaled]
X_test_bias = np.c_[np.ones((X_test_scaled.shape[0], 1)), X_test_scaled]

print(f"X_train_bias shape: {X_train_bias.shape}")

## 7. Implement Linear Regression using Gradient Descent

A vectorized, multivariate Batch Gradient Descent implementation.

In [ ]:
def compute_cost(theta, X, y):
    m = len(y)
    predictions = X.dot(theta)
    return (1 / (2 * m)) * np.sum((predictions - y) ** 2)


def batch_gradient_descent(X, y, learning_rate=0.05, iterations=1000):
    m, n = X.shape
    theta = np.zeros(n)
    cost_history = []

    for i in range(iterations):
        predictions = X.dot(theta)
        error = predictions - y
        gradients = (1 / m) * X.T.dot(error)
        theta = theta - learning_rate * gradients
        cost_history.append(compute_cost(theta, X, y))

    return theta, cost_history

In [ ]:
learning_rate = 0.1
iterations = 1000

theta, cost_history = batch_gradient_descent(
    X_train_bias, y_train, learning_rate=learning_rate, iterations=iterations
)

print(f"Final training cost: {cost_history[-1]:.4f}")
print(f"Intercept: {theta[0]:.4f}")

## 8. Effect of Learning Rate on Convergence

Different learning rates are compared over a fixed number of iterations to
observe their effect on convergence speed and stability.

In [ ]:
learning_rates = [0.001, 0.01, 0.05, 0.1, 0.3, 0.5]
lr_histories = {}

for lr in learning_rates:
    _, history = batch_gradient_descent(X_train_bias, y_train, learning_rate=lr, iterations=200)
    lr_histories[lr] = history

plt.figure(figsize=(9, 6))
for lr, history in lr_histories.items():
    plt.plot(history, label=f"lr = {lr}")

plt.xlabel("Iterations")
plt.ylabel("Cost (MSE)")
plt.title("Effect of Learning Rate on Convergence")
plt.legend()
plt.ylim(0, lr_histories[learning_rates[0]][0] * 1.1)
plt.show()

Very small learning rates (e.g. `0.001`) converge slowly and would need many
more iterations to reach the minimum. Moderate rates (`0.05` - `0.3`) converge
quickly and smoothly. Very large learning rates (e.g. `0.5`) can overshoot the
minimum and oscillate or diverge, which is visible as a spike or instability in
the corresponding curve.

## 9. Loss (Cost) vs Iterations - Chosen Model

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(cost_history)
plt.xlabel("Iterations")
plt.ylabel("Loss (MSE)")
plt.title(f"Loss Curve (learning_rate = {learning_rate})")
plt.show()

## 10. Model Evaluation on Test Set

In [ ]:
y_pred = X_test_bias.dot(theta)

mae = np.mean(np.abs(y_test - y_pred))
mse = np.mean((y_test - y_pred) ** 2)
rmse = np.sqrt(mse)

ss_res = np.sum((y_test - y_pred) ** 2)
ss_tot = np.sum((y_test - np.mean(y_test)) ** 2)
r2 = 1 - (ss_res / ss_tot)

print(f"MAE  : {mae:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R2   : {r2:.4f}")

In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(y_test, y_pred, alpha=0.6)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
plt.plot(lims, lims, color="red", linestyle="--", label="Ideal fit")
plt.xlabel("Actual G3")
plt.ylabel("Predicted G3")
plt.title("Predicted vs Actual Final Grade")
plt.legend()
plt.show()

## 11. Interpretation of Results

**Convergence behaviour:** With `learning_rate = 0.1`, the training cost drops
from an initial MSE of ~128 (theta = 0) to 1.41 within 1000 iterations, with
almost all of the improvement happening in the first ~100 iterations before
the curve flattens - Batch Gradient Descent has converged for this feature set.
The learning-rate sweep over 200 iterations shows the classic trade-off:
`lr = 0.001` is still decreasing slowly and has not converged in the given
budget, `lr = 0.01`-`0.3` converge smoothly and reach a low cost within
~50-100 iterations, and `lr = 0.5` is on the edge of instability, converging
far more slowly and unevenly than the moderate rates because each step
overshoots the minimum along several dimensions before correcting.

**Prediction performance:** On the held-out test set the model achieves
MAE = 1.65, MSE = 5.66, RMSE = 2.38 and R2 = 0.72 - i.e. the model explains
about 72% of the variance in the final grade (`G3`), and predictions are on
average within about 1.6-2.4 grade points (on the 0-20 scale) of the true
value. This performance is driven largely by `G1` and `G2` (the earlier
period grades), which are strongly correlated with `G3`; the remaining
demographic/social/academic features contribute the rest. Overall, gradient
descent successfully finds a set of weights that generalizes reasonably well
from the training set to the test set, confirming that Linear Regression is a
reasonable model for this dataset once categorical features are encoded and
inputs are scaled.